# Clinical AI Workshop Notebook

This notebook brings together teaching activities for clinical vision-language models (VLMs), multimodal information, and structured clinical-note extraction. It is designed for participants with limited programming experience. Read each short explanation, run the code cell beneath it, and discuss the result before moving on.

> **Data and scope:** This workshop uses synthetic clinical records and images from the public MedMNIST dataset. Any case–image pairings presented as a single scenario are hypothetical teaching scenarios, not linked real-patient data. Model outputs may be inaccurate or overconfident, so review claims against their sources. This is a technology demonstration, not clinical guidance or validation.

The complete notebook has more activities than may fit into a live session. Facilitators can select activities for the workshop; running every model-backed cell from top to bottom makes approximately 26 requests to the local model and may take some time. Model responses can vary between runs.

## Before you start

1. Open this notebook from the repository root.
2. Install the project dependencies with `uv sync`.
3. Start Ollama and make sure `medgemma1.5:latest` is available (`ollama pull medgemma1.5:latest`).
4. Run the notebook cells in order. Image previews and generated review artifacts are written under the ignored `data/test/` folder.

The notebook contains the code needed for every activity. Run the cells in order; preview, model-response, and review steps are separated so you can inspect each stage.


## Notebook setup

The next cell imports the tools used throughout the notebook, sets the data paths, and checks that the prepared examples are available. It does not contact Ollama.


In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Any
import base64
import mimetypes
import matplotlib.pyplot as plt
from openai import OpenAI
import re
import shutil
import random
from PIL import Image as PILImage
from IPython.display import Image, display

ROOT_DIRECTORY = Path.cwd()
DATA_DIRECTORY = ROOT_DIRECTORY / "data"
OUTPUT_DIRECTORY = DATA_DIRECTORY / "test"
FHIR_DIRECTORY = DATA_DIRECTORY / "synthea_output" / "fhir"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL_NAME = "medgemma1.5:latest"

if not (ROOT_DIRECTORY / "workshop.ipynb").exists():
    raise RuntimeError("Open this notebook with the repository root as the working directory.")

required_datasets = [DATA_DIRECTORY / "pneumoniamnist", DATA_DIRECTORY / "retinamnist"]
missing_datasets = [path.name for path in required_datasets if not list(path.glob("*.jpg"))]
if missing_datasets:
    raise FileNotFoundError("Prepared image folders are empty: " + ", ".join(missing_datasets))

if not list(FHIR_DIRECTORY.glob("*.json")):
    raise FileNotFoundError(f"No synthetic FHIR bundle JSON files found in {FHIR_DIRECTORY}")

print("Setup checks passed. Ollama will be used when an activity requests a model response.")


## Clinical vision-language activities

The first group explores image descriptions, classification, evidence, uncertainty, and ways of combining image and text. Treat outputs as model responses to discuss, not as ground truth.


## Activity 1 — Describe visible features in a medical image

Separating what is visible from what is inferred helps clinicians judge whether an interpretation is grounded in the image.


In [ ]:
activity_01_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

def activity_01_find_example_image() -> Path:
    """Find the prepared pneumonia example without displaying its label filename."""
    matches = sorted(DATA_DIRECTORY.glob(activity_01_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(
            f"No prepared image matches {activity_01_IMAGE_PATTERN!r} under {DATA_DIRECTORY}."
        )
    return matches[0]

def activity_01_save_example_image(image_path: Path) -> Path:
    """Save a neutral image preview to disk and close the Matplotlib figure."""
    with PILImage.open(image_path) as image:
        figure, axis = plt.subplots(figsize=(6, 6))
        axis.imshow(image, cmap="gray")
        axis.set_title("Teaching image")
        axis.axis("off")
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_01_image_description_example.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_01_describe_image(image_path: Path) -> str:
    """Ask for visible features only, before requesting any classification."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Describe only what is visibly present in this educational "
                            "medical image. Use plain language. Separate observations "
                            "from interpretations, and mention image limitations. Do "
                            "not provide a diagnosis or treatment recommendation."
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."


In [ ]:
image_01 = activity_01_find_example_image()
preview_01 = activity_01_save_example_image(image_01)
display(Image(filename=str(preview_01)))


In [ ]:
response_01 = activity_01_describe_image(image_01)
print(response_01)


## Activity 2 — Classify medical images

Classification is a common AI task. Comparing your guess with the model’s can help reveal where its answers may differ from yours; dataset labels are for this exercise, not a clinical reference.


In [ ]:
activity_02_CLASSIFICATION_EXAMPLES = [
    {
        "title": "Chest X-ray",
        "pattern": "pneumoniamnist/11_*.jpg",
        "choices": "normal, pneumonia",
    },
    {
        "title": "Retinal image",
        "pattern": "retinamnist/181_*.jpg",
        "choices": (
            "no diabetic retinopathy, mild diabetic retinopathy, "
            "moderate diabetic retinopathy, severe diabetic retinopathy, "
            "proliferative diabetic retinopathy"
        ),
    },
]

def activity_02_load_classification_examples() -> list[dict[str, str | Path]]:
    """Resolve one prepared image for each classification task."""
    examples = []
    for example in activity_02_CLASSIFICATION_EXAMPLES:
        matches = sorted(DATA_DIRECTORY.glob(str(example["pattern"])))
        if not matches:
            raise FileNotFoundError(
                f"No image matches {example['pattern']!r} under {DATA_DIRECTORY}."
            )
        examples.append({**example, "image_path": matches[0]})
    return examples

def activity_02_save_classification_examples(examples: list[dict[str, str | Path]]) -> Path:
    """Save a neutral preview grid without opening an interactive figure."""
    figure, axes = plt.subplots(1, len(examples), figsize=(6 * len(examples), 5))
    if len(examples) == 1:
        axes = [axes]
    for axis, example in zip(axes, examples):
        with PILImage.open(Path(example["image_path"])) as image:
            axis.imshow(image, cmap="gray")
        axis.set_title(str(example["title"]))
        axis.axis("off")
    figure.tight_layout()
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_02_classification_examples.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_02_classify_image(image_path: Path, choices: str, task: str) -> str:
    """Request a single image classification from the model."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            f"Task: {task}. Choose the single best category from: "
                            f"{choices}. Reply with the category name and a brief "
                            "note if the image is insufficient for classification."
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_02_classify_all_examples(
    examples: list[dict[str, str | Path]],
) -> list[dict[str, str]]:
    """Run both image tasks and return neutral titles with model responses."""
    results = []
    for example in examples:
        task = (
            "classify this chest X-ray"
            if example["title"] == "Chest X-ray"
            else "classify the diabetic retinopathy grade in this retinal image"
        )
        answer = activity_02_classify_image(
            Path(example["image_path"]), str(example["choices"]), task
        )
        results.append({"title": str(example["title"]), "response": answer})
    return results

def activity_02_display_classification_results(results: list[dict[str, str]]) -> None:
    """Print the model responses without revealing the known labels."""
    for result in results:
        print(f"{result['title']} — model response:\n{result['response']}\n")


In [ ]:
examples_02 = activity_02_load_classification_examples()
preview_02 = activity_02_save_classification_examples(examples_02)
display(Image(filename=str(preview_02)))


In [ ]:
results_02 = activity_02_classify_all_examples(examples_02)
activity_02_display_classification_results(results_02)


**Pause:** Compare your guesses with the model responses. When ready, run the next cell to reveal the dataset labels.

In [ ]:
print("Dataset labels (for comparison, not a clinical reference):")
print("Chest X-ray: pneumonia")
print("Retinal image: moderate diabetic retinopathy")


## Activity 3 — Ask a focused visual question

A focused question can keep the response relevant to a clinician’s task and reduce unrelated or unsupported details.


In [ ]:
activity_03_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

activity_03_QUESTION = "What broad anatomical structures are visible? Describe only visible features."

def activity_03_find_example_image() -> Path:
    """Find one prepared chest X-ray example."""
    matches = sorted(DATA_DIRECTORY.glob(activity_03_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_03_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_03_save_image_and_question_preview(image_path: Path, question: str) -> Path:
    """Save a neutral image preview and print the question for discussion."""
    with PILImage.open(image_path) as image:
        figure, axis = plt.subplots(figsize=(6, 6))
        axis.imshow(image, cmap="gray")
        axis.set_title("Teaching image")
        axis.axis("off")
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_03_visual_question_example.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image preview to: {output_path}")
    print(f"Question: {question}")
    return output_path

def activity_03_ask_visual_question(image_path: Path, question: str) -> str:
    """Send an image and a focused question to MedGemma."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."


In [ ]:
image_03 = activity_03_find_example_image()
preview_03 = activity_03_save_image_and_question_preview(image_03, activity_03_QUESTION)
display(Image(filename=str(preview_03)))


In [ ]:
response_03 = activity_03_ask_visual_question(image_03, activity_03_QUESTION)
print(response_03)


## Activity 4 — Support interpretations with visible evidence

Connecting an interpretation to visible features makes it easier to review what the image supports and what remains uncertain.


In [ ]:
activity_04_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

activity_04_PROMPT = (
    "For this educational image, state a possible classification and list the "
    "visible features that support or do not support it. Separate observations "
    "from interpretation. State what is uncertain and what cannot be determined "
    "from this image alone. Do not provide treatment advice."
)

def activity_04_find_example_image() -> Path:
    """Find one prepared image for the evidence-linked interpretation activity."""
    matches = sorted(DATA_DIRECTORY.glob(activity_04_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_04_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_04_save_example_image(image_path: Path) -> Path:
    """Save an image preview and close its Matplotlib figure."""
    with PILImage.open(image_path) as image:
        figure, axis = plt.subplots(figsize=(6, 6))
        axis.imshow(image, cmap="gray")
        axis.set_title("Teaching image")
        axis.axis("off")
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_04_evidence_interpretation_example.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_04_interpret_with_evidence(image_path: Path, prompt: str = activity_04_PROMPT) -> str:
    """Request a classification with visible evidence and explicit limitations."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."


In [ ]:
image_04 = activity_04_find_example_image()
preview_04 = activity_04_save_example_image(image_04)
display(Image(filename=str(preview_04)))


In [ ]:
response_04 = activity_04_interpret_with_evidence(image_04)
print(response_04)


## Activity 5 — Request structured image findings

Headings can make findings easier to scan and organize. Comparing the original response with the fields created by a script also shows that a clear format does not guarantee clinical accuracy.

The model first returns findings under headings. A script then sorts the text under those headings into fields and flags any requested fields it cannot find. It checks the format, not whether the findings are correct.


In [ ]:
activity_05_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

def activity_05_find_example_image() -> Path:
    """Find one prepared image for the structured findings activity."""
    matches = sorted(DATA_DIRECTORY.glob(activity_05_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_05_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_05_save_example_image(image_path: Path) -> Path:
    """Save an image preview and close its Matplotlib figure."""
    with PILImage.open(image_path) as image:
        figure, axis = plt.subplots(figsize=(6, 6))
        axis.imshow(image, cmap="gray")
        axis.set_title("Teaching image")
        axis.axis("off")
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_05_structured_findings_example.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_05_extract_structured_findings(image_path: Path) -> str:
    """Ask for findings using labeled plain text rather than a strict JSON schema."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Use these plain-text headings exactly, each on its own "
                            "line: Image type:, Visible findings:, Possible "
                            "interpretation:, Uncertainty:, Cannot determine:. Put "
                            "findings and unknowns on separate bullet lines. Use only "
                            "features visible in the image. State uncertainty. Do "
                            "not provide treatment advice."
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    response_text = response.choices[0].message.content or ""
    if "<unused95>" in response_text:
        response_text = response_text.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return response_text or "The model returned an empty response."

def activity_05_parse_labeled_findings(response_text: str) -> dict[str, Any]:
    """Parse known headings and bullet lists, retaining the original response."""
    heading_map = {
        "image type": "image_type",
        "visible findings": "visible_findings",
        "possible interpretation": "possible_interpretation",
        "uncertainty": "uncertainty",
        "cannot determine": "cannot_determine",
    }
    list_fields = {"visible_findings", "cannot_determine"}
    findings: dict[str, Any] = {
        "image_type": "",
        "visible_findings": [],
        "possible_interpretation": "",
        "uncertainty": "",
        "cannot_determine": [],
        "raw_response": response_text,
    }
    current_field = ""
    for line in response_text.splitlines():
        heading = re.match(r"^\s*([^:]+):\s*(.*)$", line)
        if heading and heading.group(1).casefold().strip() in heading_map:
            current_field = heading_map[heading.group(1).casefold().strip()]
            value = heading.group(2).strip()
            if value:
                if current_field in list_fields:
                    findings[current_field].append(value.lstrip("-* "))
                else:
                    findings[current_field] = value
            continue

        value = line.strip().lstrip("-*• ").strip()
        if not value or not current_field:
            continue
        if current_field in list_fields:
            findings[current_field].append(value)
        elif not findings[current_field]:
            findings[current_field] = value
        else:
            findings[current_field] += f" {value}"
    return findings

def activity_05_validate_structured_findings(findings: dict[str, Any]) -> list[str]:
    """Flag labeled fields that are absent or empty after plain-text parsing."""
    expected_fields = (
        "image_type",
        "visible_findings",
        "possible_interpretation",
        "uncertainty",
        "cannot_determine",
    )
    return [
        f"Could not parse a non-empty '{field_name}' field from the response."
        for field_name in expected_fields
        if not findings.get(field_name)
    ]


In [ ]:
image_05 = activity_05_find_example_image()
preview_05 = activity_05_save_example_image(image_05)
display(Image(filename=str(preview_05)))


In [ ]:
raw_05 = activity_05_extract_structured_findings(image_05)
print(raw_05)


In [ ]:
parsed_05 = activity_05_parse_labeled_findings(raw_05)
print(json.dumps(parsed_05, indent=2))
issues_05 = activity_05_validate_structured_findings(parsed_05)
print(issues_05 or "All requested fields were parsed.")


## Activity 6 — Compare two images from different people

Differences between images of different people do not show change over time. Keeping that distinction clear helps avoid mistaken conclusions about progression or a shared patient.


In [ ]:
activity_06_IMAGE_PATTERNS = [
    "pneumoniamnist/28_*.jpg",
    "pneumoniamnist/11_*.jpg",
]

def activity_06_find_comparison_images() -> list[Path]:
    """Find one prepared example from each chest X-ray label group."""
    image_paths = []
    for pattern in activity_06_IMAGE_PATTERNS:
        matches = sorted(DATA_DIRECTORY.glob(pattern))
        if not matches:
            raise FileNotFoundError(f"No prepared image matches {pattern!r}.")
        image_paths.append(matches[0])
    return image_paths

def activity_06_save_comparison_images(image_paths: list[Path]) -> Path:
    """Save a neutral comparison grid and close the Matplotlib figure."""
    figure, axes = plt.subplots(1, len(image_paths), figsize=(10, 5))
    for axis, image_path, title in zip(axes, image_paths, ("Example A", "Example B")):
        with PILImage.open(image_path) as image:
            axis.imshow(image, cmap="gray")
        axis.set_title(title)
        axis.axis("off")
    figure.tight_layout()
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_06_comparison_examples.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image comparison to: {output_path}")
    return output_path

def activity_06_compare_images(image_paths: list[Path]) -> str:
    """Ask the model to describe similarities and differences between images."""
    content: list[dict[str, str | dict[str, str]]] = [
        {
            "type": "text",
            "text": (
                "Compare Example A and Example B. Describe visible similarities "
                "and differences only. These are different teaching examples, not "
                "images of the same person. Do not infer progression or recommend "
                "treatment."
            ),
        }
    ]
    for title, image_path in zip(("Example A", "Example B"), image_paths):
        mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
        encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
        content.append({"type": "text", "text": title})
        content.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:{mime_type};base64,{encoded_image}"},
            }
        )
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[{"role": "user", "content": content}],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."


In [ ]:
images_06 = activity_06_find_comparison_images()
preview_06 = activity_06_save_comparison_images(images_06)
display(Image(filename=str(preview_06)))


In [ ]:
response_06 = activity_06_compare_images(images_06)
print(response_06)


## Activity 7 — Add clinical context to image interpretation

Clinical history can affect how an image is interpreted. Keeping reported details separate from visible features helps clinicians see what each source contributes and what remains uncertain.


In [ ]:
activity_07_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

CASE_RECORD = """
An adult presents with a three-day cough and fever. Oxygen saturation is
91% on room air.
""".strip()

def activity_07_find_example_image() -> Path:
    """Find one prepared chest X-ray example for the supplied case."""
    matches = sorted(DATA_DIRECTORY.glob(activity_07_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_07_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_07_save_image_and_context_preview(image_path: Path, context: str) -> Path:
    """Save the image preview; print the supplied case details separately."""
    with PILImage.open(image_path) as image:
        figure, axis = plt.subplots(figsize=(6, 6))
        axis.imshow(image, cmap="gray")
        axis.set_title("Teaching image")
        axis.axis("off")
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_07_contextualized_case_image.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image preview to: {output_path}")
    print("Clinical case details supplied to the model:")
    print(context)
    return output_path

def activity_07_interpret_with_context(image_path: Path, context: str) -> str:
    """Ask how the supplied context affects image interpretation, separating sources."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Interpret the image in the context of the supplied clinical record. "
                            "Describe visible observations, identify relevant case details, "
                            "and explain how the combined information affects interpretation "
                            "and uncertainty. Do not provide treatment instructions.\n\n"
                            f"Clinical record:\n{context}"
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."


In [ ]:
image_07 = activity_07_find_example_image()
preview_07 = activity_07_save_image_and_context_preview(image_07, CASE_RECORD)
display(Image(filename=str(preview_07)))


In [ ]:
response_07 = activity_07_interpret_with_context(image_07, CASE_RECORD)
print(response_07)


## Activity 8 — Compare image-only, record-only, and combined inputs

Comparing the sources separately and together shows how each affects the response. It can also reveal whether the model blurs which source supports a statement.


In [ ]:
image_matches_08 = sorted(DATA_DIRECTORY.glob("pneumoniamnist/11_*.jpg"))
if not image_matches_08:
    raise FileNotFoundError("The prepared chest X-ray example was not found.")
image_08 = image_matches_08[0]
record_08 = CASE_RECORD
with PILImage.open(image_08) as image:
    figure, axis = plt.subplots(figsize=(6, 6))
    axis.imshow(image, cmap="gray")
    axis.set_title("Teaching image")
    axis.axis("off")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
preview_08 = OUTPUT_DIRECTORY / "activity_08_paired_case_image.png"
figure.savefig(preview_08, dpi=150, bbox_inches="tight")
plt.close(figure)
display(Image(filename=str(preview_08)))
print("Case record:")
print(record_08)

In [ ]:
mime_type_08 = mimetypes.guess_type(image_08.name)[0] or "image/jpeg"
encoded_image_08 = base64.b64encode(image_08.read_bytes()).decode("utf-8")
client_08_image = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
response_08_image = client_08_image.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.0,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe visible features in this chest image and state what is uncertain."},
            {"type": "image_url", "image_url": {"url": f"data:{mime_type_08};base64,{encoded_image_08}"}},
        ],
    }],
)
image_only_08 = response_08_image.choices[0].message.content or ""
if "<unused95>" in image_only_08:
    image_only_08 = image_only_08.rsplit("<unused95>", maxsplit=1)[-1].strip()
print("Image only:")
print(image_only_08)

In [ ]:
client_08_record = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
response_08_record = client_08_record.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.0,
    messages=[{
        "role": "user",
        "content": "Summarize the supplied clinical record, keeping to the details it states.\n\n" + record_08,
    }],
)
record_only_08 = response_08_record.choices[0].message.content or ""
if "<unused95>" in record_only_08:
    record_only_08 = record_only_08.rsplit("<unused95>", maxsplit=1)[-1].strip()
print("Record only:")
print(record_only_08)

In [ ]:
client_08_combined = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
response_08_combined = client_08_combined.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.0,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "Interpret this chest image in the context of the supplied clinical record. "
                    "Distinguish visible image observations from reported case details, explain "
                    "how the combined information affects your interpretation, and state what "
                    "remains uncertain. Do not provide treatment instructions.\n\n"
                    f"Clinical record:\n{record_08}"
                ),
            },
            {"type": "image_url", "image_url": {"url": f"data:{mime_type_08};base64,{encoded_image_08}"}},
        ],
    }],
)
combined_08 = response_08_combined.choices[0].message.content or ""
if "<unused95>" in combined_08:
    combined_08 = combined_08.rsplit("<unused95>", maxsplit=1)[-1].strip()
print("Image and case record:")
print(combined_08)

## Activity 9 — Check whether the image and record belong to the same patient

The image and synthetic record were selected independently, with no shared patient identifier. Without confirmation that they belong to the same person, a patient-level comparison cannot be made.

The notebook turns selected entries from a synthetic health record into a short summary, then selects a public image separately. It gives both to the model and asks it to describe them without assuming they belong to the same patient.


In [ ]:
def activity_09_summary_find_fhir_bundle() -> Path:
    """Choose the first patient bundle in a stable, repeatable order."""
    for bundle_path in sorted(FHIR_DIRECTORY.glob("*.json")):
        with bundle_path.open(encoding="utf-8") as file:
            bundle = json.load(file)
        if bundle.get("resourceType") == "Bundle":
            return bundle_path

    raise FileNotFoundError(f"No FHIR Bundle JSON files found in {FHIR_DIRECTORY}")

def activity_09_summary_read_resources(bundle_path: Path) -> list[dict[str, Any]]:
    """Read the resource entries from one FHIR Bundle."""
    with bundle_path.open(encoding="utf-8") as file:
        bundle = json.load(file)

    if bundle.get("resourceType") != "Bundle":
        raise ValueError(f"Expected a FHIR Bundle in {bundle_path}")

    # A Bundle groups resources together. Each entry contains one resource.
    return [
        entry["resource"]
        for entry in bundle.get("entry", [])
        if isinstance(entry.get("resource"), dict)
    ]

def activity_09_summary_first_code_display(resource: dict[str, Any]) -> str:
    """Get a human-readable code label from common FHIR fields."""
    for field_name in ("code", "medicationCodeableConcept", "type", "class"):
        field_value = resource.get(field_name, {})
        # Some FHIR fields (such as Encounter.type) contain a list of concepts.
        concepts = field_value if isinstance(field_value, list) else [field_value]
        for concept in concepts:
            if not isinstance(concept, dict):
                continue
            # Some resources use a Code directly rather than a CodeableConcept.
            if concept.get("display"):
                return str(concept["display"])
            if concept.get("text"):
                return str(concept["text"])
            for coding in concept.get("coding", []):
                if coding.get("display"):
                    return str(coding["display"])
    return "Description not provided"

def activity_09_summary_describe_observation(observation: dict[str, Any]) -> str:
    """Format an observation's name and value for a human reader."""
    description = activity_09_summary_first_code_display(observation)
    if "valueQuantity" in observation:
        quantity = observation["valueQuantity"]
        value = quantity.get("value", "")
        unit = quantity.get("unit", "")
        return f"{description}: {value} {unit}".strip()
    if "valueString" in observation:
        return f"{description}: {observation['valueString']}"
    if "valueCodeableConcept" in observation:
        return f"{description}: {activity_09_summary_first_code_display({'code': observation['valueCodeableConcept']})}"
    return description

def activity_09_summary_build_patient_summary(resources: list[dict[str, Any]]) -> str:
    """Turn selected FHIR resources into a short, readable teaching summary."""
    resource_counts = Counter(
        resource.get("resourceType", "Unknown") for resource in resources
    )
    patient = next(
        (resource for resource in resources if resource.get("resourceType") == "Patient"),
        {},
    )

    # Names and identifiers are intentionally left out of the teaching summary.
    lines = ["Synthetic patient summary", "(Generated data for education only)"]
    if patient.get("gender"):
        lines.append(f"Administrative gender: {patient['gender']}")
    if patient.get("birthDate"):
        lines.append(f"Birth date: {patient['birthDate']}")

    lines.append("\nFHIR resource counts:")
    for resource_type, count in sorted(resource_counts.items()):
        lines.append(f"- {resource_type}: {count}")

    for resource_type, heading in (
        ("Condition", "Conditions"),
        ("MedicationRequest", "Medication requests"),
        ("AllergyIntolerance", "Allergies"),
        ("Encounter", "Encounters"),
        ("Observation", "Observations"),
    ):
        matching_resources = [
            resource
            for resource in resources
            if resource.get("resourceType") == resource_type
        ]
        if not matching_resources:
            continue

        lines.append(f"\n{heading} (showing up to 5):")
        for resource in matching_resources[:5]:
            if resource_type == "Observation":
                description = activity_09_summary_describe_observation(resource)
            else:
                description = activity_09_summary_first_code_display(resource)
            lines.append(f"- {description}")

    return "\n".join(lines)

def activity_09_summary_summarize_one_patient() -> Path:
    """Read one Synthea bundle, print its summary, and save it under data/test."""
    bundle_path = activity_09_summary_find_fhir_bundle()
    resources = activity_09_summary_read_resources(bundle_path)
    summary = activity_09_summary_build_patient_summary(resources)

    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "synthea_patient_summary.txt"
    output_path.write_text(summary + "\n", encoding="utf-8")
    print(f"Source bundle: {bundle_path.name}")
    print(summary)
    print(f"\nSaved summary to: {output_path}")
    return output_path

In [ ]:
summary_path_09 = activity_09_summary_summarize_one_patient()
record_09 = summary_path_09.read_text(encoding="utf-8")
image_matches_09 = sorted(DATA_DIRECTORY.glob("pneumoniamnist/11_*.jpg"))
if not image_matches_09:
    raise FileNotFoundError("The prepared chest X-ray example was not found.")
image_09 = image_matches_09[0]
print("Synthetic record, independently selected:")
print(record_09)
with PILImage.open(image_09) as source_image:
    figure, axis = plt.subplots(figsize=(6, 6))
    axis.imshow(source_image, cmap="gray")
    axis.set_title("Independently selected teaching image")
    axis.axis("off")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
preview_09 = OUTPUT_DIRECTORY / "activity_09_unlinked_consistency_image.png"
figure.savefig(preview_09, dpi=150, bbox_inches="tight")
plt.close(figure)
display(Image(filename=str(preview_09)))

In [ ]:
mime_type_09 = mimetypes.guess_type(image_09.name)[0] or "image/jpeg"
encoded_image_09 = base64.b64encode(image_09.read_bytes()).decode("utf-8")
client_09 = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
response_09 = client_09.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.0,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "The supplied clinical record and image were selected independently and "
                    "have no shared patient identifier. Describe record facts and visible image "
                    "features separately. State whether patient-level consistency can be "
                    "assessed from these inputs, and do not combine them into a diagnosis.\n\n"
                    f"Clinical record:\n{record_09}"
                ),
            },
            {"type": "image_url", "image_url": {"url": f"data:{mime_type_09};base64,{encoded_image_09}"}},
        ],
    }],
)
answer_09 = response_09.choices[0].message.content or ""
if "<unused95>" in answer_09:
    answer_09 = answer_09.rsplit("<unused95>", maxsplit=1)[-1].strip()
print(answer_09)

## Activity 10 — Test whether added context is relevant

Irrelevant personal details may influence a model’s response. Comparing answers with and without those details helps show whether the model is relying on information that should not affect the interpretation.

The image and question stay the same in both requests; only the case details change. This makes it easier to see whether adding personal details affects the response.


In [ ]:
activity_10_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

activity_10_RELEVANT_CONTEXT = (
    "Clinical note: the person has had a cough for three days and "
    "reports fever."
)

activity_10_CONTEXT_WITH_INCIDENTAL_DETAILS = (
    "Clinical note: the person has had a cough for three days and "
    "reports fever. The person works as a librarian and enjoys gardening."
)

def activity_10_find_example_image() -> Path:
    """Find one prepared image to use with both context variants."""
    matches = sorted(DATA_DIRECTORY.glob(activity_10_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_10_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_10_save_image_and_print_contexts(image_path: Path) -> Path:
    """Save the shared image preview and print both context variants."""
    with PILImage.open(image_path) as image:
        figure, axis = plt.subplots(figsize=(6, 6))
        axis.imshow(image, cmap="gray")
        axis.set_title("Same teaching image for both requests")
        axis.axis("off")
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_10_context_relevance_image.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image preview to: {output_path}")
    print(f"Relevant context only:\n{activity_10_RELEVANT_CONTEXT}\n")
    print(f"Same context plus incidental details:\n{activity_10_CONTEXT_WITH_INCIDENTAL_DETAILS}")
    return output_path

def activity_10_ask_with_context(image_path: Path, context: str) -> str:
    """Ask for image observations and keep non-image context identified as text."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Describe visible image features and interpret them in "
                            "the context of the supplied clinical details. Distinguish "
                            "observations from reported history and state uncertainty.\n\n"
                            f"Case information:\n{context}"
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_10_compare_context_variants(image_path: Path) -> dict[str, str]:
    """Run the same task with relevant context, then with incidental details added."""
    return {
        "Relevant context": activity_10_ask_with_context(image_path, activity_10_RELEVANT_CONTEXT),
        "Relevant plus incidental context": activity_10_ask_with_context(
            image_path, activity_10_CONTEXT_WITH_INCIDENTAL_DETAILS
        ),
    }

def activity_10_display_context_responses(responses: dict[str, str]) -> None:
    """Print the two outputs for participant comparison."""
    for heading, response in responses.items():
        print(f"{heading}:\n{response}\n")


In [ ]:
image_10 = activity_10_find_example_image()
preview_10 = activity_10_save_image_and_print_contexts(image_10)
display(Image(filename=str(preview_10)))


In [ ]:
relevant_response_10 = activity_10_ask_with_context(image_10, activity_10_RELEVANT_CONTEXT)
print("Relevant context:", relevant_response_10)


In [ ]:
incidental_response_10 = activity_10_ask_with_context(image_10, activity_10_CONTEXT_WITH_INCIDENTAL_DETAILS)
print("Relevant plus incidental context:", incidental_response_10)


## Activity 11 — Generate follow-up questions

Follow-up questions can help identify what information a clinician may still need, without presenting missing details as facts. Comparing the full response with the questions a script extracts also shows how formatting can affect what is captured.

The notebook keeps the model’s full response, then a script collects only lines beginning with `QUESTION:`. Compare both: questions may be missed if the model uses a different format.


In [ ]:
activity_11_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

activity_11_PROMPT = (
    "Give 3 to 5 concise clinician follow-up questions to help interpret this "
    "image. Put one question on each line, each starting with 'QUESTION:'. Do "
    "not provide image findings, an impression, a diagnosis, or treatment advice."
)

def activity_11_find_example_image() -> Path:
    """Find one prepared chest X-ray example."""
    matches = sorted(DATA_DIRECTORY.glob(activity_11_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_11_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_11_save_image_preview(image_path: Path) -> Path:
    """Copy the example to an unlabeled teaching-preview filename."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_11_follow_up_questions_image.jpg"
    output_path.write_bytes(image_path.read_bytes())
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_11_generate_follow_up_questions(image_path: Path, prompt: str = activity_11_PROMPT) -> str:
    """Ask for relevant follow-up questions without inferring missing history."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_11_parse_follow_up_questions(response_text: str) -> list[str]:
    """Extract QUESTION-labeled lines while retaining the raw text separately."""
    questions = []
    for line in response_text.splitlines():
        match = re.match(r"^\s*(?:[-*•]\s*|\d+[.)]\s*)?QUESTION\s*:\s*(.*)$", line, re.I)
        if match and match.group(1).strip():
            questions.append(match.group(1).strip())
    return questions

def activity_11_save_follow_up_response(response_text: str, questions: list[str]) -> Path:
    """Save both the raw model response and questions parsed from its labels."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_11_follow_up_questions.json"
    result = {"raw_response": response_text, "parsed_questions": questions}
    output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("Parsed follow-up questions:")
    for question in questions:
        print(f"- {question}")
    if not questions:
        print("No QUESTION-labeled lines were parsed; inspect the raw response.")
    print(f"Saved response to: {output_path}")
    return output_path


In [ ]:
image_11 = activity_11_find_example_image()
preview_11 = activity_11_save_image_preview(image_11)
display(Image(filename=str(preview_11)))


In [ ]:
raw_11 = activity_11_generate_follow_up_questions(image_11)
print(raw_11)


In [ ]:
questions_11 = activity_11_parse_follow_up_questions(raw_11)
activity_11_save_follow_up_response(raw_11, questions_11)


## Activity 12 — Draft a summary for another clinician to review

A useful summary keeps case details, image observations, and uncertainty distinct so they can be checked separately.

The model receives the image and the case details used earlier and drafts a summary with separate sections for the case, image observations, uncertainty, and items to check. The notebook saves the draft for review; it does not complete a real handoff.


In [ ]:
activity_12_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

# Reuse the case record introduced in Activity 7.

def activity_12_find_example_image() -> Path:
    """Find one prepared chest X-ray example."""
    matches = sorted(DATA_DIRECTORY.glob(activity_12_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_12_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_12_save_image_preview(image_path: Path) -> Path:
    """Copy the image under a label-neutral teaching filename."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_12_handoff_teaching_image.jpg"
    output_path.write_bytes(image_path.read_bytes())
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_12_create_clinical_handoff(image_path: Path, context: str) -> str:
    """Request a concise draft handoff with source facts kept distinct."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Draft a short clinician-facing teaching handoff with "
                            "sections for (1) facts supplied in the vignette, (2) "
                            "visible image observations, (3) uncertainty or missing "
                            "information, and (4) items for clinician review. In "
                            "the review section, list only verification questions "
                            "such as confirming source linkage or consulting prior "
                            "images. Do not suggest diagnoses, tests, management, "
                            "or treatment. Do not treat "
                            "image observations as confirmed diagnoses.\n\n"
                            f"Clinical record:\n{context}"
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_12_save_handoff(handoff: str) -> Path:
    """Save and print the draft handoff."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_12_clinical_handoff_draft.txt"
    output_path.write_text(handoff + "\n", encoding="utf-8")
    print(handoff)
    print(f"Saved draft to: {output_path}")
    return output_path


In [ ]:
image_12 = activity_12_find_example_image()
preview_12 = activity_12_save_image_preview(image_12)
display(Image(filename=str(preview_12)))


In [ ]:
handoff_12 = activity_12_create_clinical_handoff(image_12, CASE_RECORD)
activity_12_save_handoff(handoff_12)


## Activity 13 — Restate a draft in patient-friendly language

Clear language can make information easier for patients to understand, but a rewrite should preserve the original meaning and uncertainty. Check that it does not add facts or imply more certainty than the draft.

This uses two model requests: first, the model describes the image; then it rewrites that description in simpler language. The second request works from the draft, so compare the rewrite with the original for changes in meaning or certainty.


In [ ]:
activity_13_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

def activity_13_find_example_image() -> Path:
    """Find one prepared image for the two-step explanation activity."""
    matches = sorted(DATA_DIRECTORY.glob(activity_13_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_13_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_13_save_image_preview(image_path: Path) -> Path:
    """Copy the image under a label-neutral teaching filename."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_13_patient_friendly_teaching_image.jpg"
    output_path.write_bytes(image_path.read_bytes())
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_13_draft_visible_findings(image_path: Path) -> str:
    """Ask for a brief image-only description to use as the rewriting source."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Describe only visible features in this teaching image. "
                            "Mention uncertainty; do not diagnose or recommend treatment."
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_13_restate_for_patient(findings: str) -> str:
    """Rewrite the supplied draft in plain language without adding new facts."""
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": (
                    "Rewrite the following draft as clear, respectful, patient-"
                    "friendly language for a teaching example. Refer to "
                    "'the image' rather than 'your image'. Preserve only information "
                    "in the draft; do not add causes, alternative explanations, a "
                    "diagnosis, prognosis, or treatment advice. Preserve uncertainty "
                    "and say the draft requires clinician review.\n\n"
                    f"Draft image description:\n{findings}"
                ),
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_13_save_patient_friendly_draft(source_findings: str, rewrite: str) -> Path:
    """Save both stages so reviewers can compare meaning and wording."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_13_patient_friendly_draft.txt"
    output_path.write_text(
        "Model-generated image description (source):\n"
        f"{source_findings}\n\n"
        "Patient-friendly draft (requires clinician review):\n"
        f"{rewrite}\n",
        encoding="utf-8",
    )
    print(rewrite)
    print(f"Saved source and rewrite to: {output_path}")
    return output_path


In [ ]:
image_13 = activity_13_find_example_image()
preview_13 = activity_13_save_image_preview(image_13)
display(Image(filename=str(preview_13)))


In [ ]:
findings_13 = activity_13_draft_visible_findings(image_13)
print(findings_13)


In [ ]:
rewrite_13 = activity_13_restate_for_patient(findings_13)
print("Patient-friendly draft (requires clinician review):", rewrite_13)
activity_13_save_patient_friendly_draft(findings_13, rewrite_13)


## Activity 14 — Check whether the model communicates uncertainty

Recognizing when an image is not sufficient for a reliable assessment is important. A confident-sounding answer does not mean the image supports a reliable grade.


In [ ]:
activity_14_IMAGE_PATTERN = "retinamnist/181_*.jpg"

activity_14_PROMPT = (
    "Assess whether this image alone supports a reliable diabetic retinopathy "
    "grade. If the image is too limited or uncertain, say that you cannot "
    "determine a grade rather than guessing. State the limitation and what "
    "additional information or review may be needed. This is a model-behavior "
    "demonstration, not a clinical assessment."
)

def activity_14_find_example_image() -> Path:
    """Find one prepared retinal example for the abstention prompt."""
    matches = sorted(DATA_DIRECTORY.glob(activity_14_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_14_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_14_save_image_preview(image_path: Path) -> Path:
    """Copy the image to a label-neutral preview filename."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_14_uncertainty_example.jpg"
    output_path.write_bytes(image_path.read_bytes())
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_14_ask_for_uncertainty(image_path: Path, prompt: str = activity_14_PROMPT) -> str:
    """Ask the model to state when the provided visual information is insufficient."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_14_save_uncertainty_response(response_text: str) -> Path:
    """Save the raw response for participant review."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_14_uncertainty_response.txt"
    output_path.write_text(response_text + "\n", encoding="utf-8")
    print(response_text)
    print(f"Saved response to: {output_path}")
    return output_path


In [ ]:
image_14 = activity_14_find_example_image()
preview_14 = activity_14_save_image_preview(image_14)
display(Image(filename=str(preview_14)))


In [ ]:
response_14 = activity_14_ask_for_uncertainty(image_14)
activity_14_save_uncertainty_response(response_14)


## Activity 15 — Review model claims against the source image

Checking each claim against the image can help identify unsupported statements before anyone relies on them. The review prompts support discussion; they do not confirm that the model is correct.

The notebook saves the image, model response, and review questions together in a file for human review. It does not automatically decide which claims are supported.


In [ ]:
activity_15_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

activity_15_REVIEW_QUESTIONS = [
    "Which response statements are directly supported by visible image features?",
    "Which statements are uncertain, speculative, or not assessable from this image?",
    "Did the response add patient history or other facts not supplied?",
    "What would need clinician or source-image review?",
]

def activity_15_find_example_image() -> Path:
    """Find one prepared chest X-ray to use in the review packet."""
    matches = sorted(DATA_DIRECTORY.glob(activity_15_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_15_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_15_generate_image_response(image_path: Path) -> str:
    """Generate a short image description that participants can audit."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Describe visible features and possible findings in this "
                            "educational image. Distinguish observations from "
                            "interpretation and state uncertainty."
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_15_save_review_packet(image_path: Path, response_text: str) -> Path:
    """Save the image, raw response, and blank human-review prompts together."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    image_copy = OUTPUT_DIRECTORY / "activity_15_claim_review_image.jpg"
    shutil.copyfile(image_path, image_copy)
    packet: dict[str, Any] = {
        "model": MODEL_NAME,
        "image": image_copy.name,
        "model_response": response_text,
        "review_questions": activity_15_REVIEW_QUESTIONS,
        "participant_notes": "",
    }
    output_path = OUTPUT_DIRECTORY / "activity_15_unsupported_claims_review.json"
    output_path.write_text(json.dumps(packet, indent=2), encoding="utf-8")
    print("Model response for human review:")
    print(response_text)
    print(f"Saved review packet to: {output_path}")
    return output_path


In [ ]:
image_15 = activity_15_find_example_image()
preview_15 = DATA_DIRECTORY / "test" / "activity_15_claim_review_image.jpg"
display(Image(filename=str(image_15)))


In [ ]:
response_15 = activity_15_generate_image_response(image_15)
print(response_15)


In [ ]:
packet_15 = activity_15_save_review_packet(image_15, response_15)
print("Review packet saved to:", packet_15)
print("Review questions:")
for question in activity_15_REVIEW_QUESTIONS:
    print("-", question)


## Activity 16 — Prioritize examples for human review

Prioritizing can help direct attention, but this unvalidated demonstration is not clinical triage. The images are from different people, and the dataset labels do not indicate clinical urgency.

The notebook selects three images, shuffles their order, and shows them as examples without their labels. The model returns an order, and a script checks that all three examples appear once. That check is about the response format, not whether the order is clinically appropriate.


In [ ]:
activity_16_RANDOM_SEED = 42

activity_16_RETINA_EXAMPLES = [
    ("Example A", "155_*.jpg"),
    ("Example B", "181_*.jpg"),
    ("Example C", "165_*.jpg"),
]

def activity_16_load_examples() -> list[dict[str, str | Path]]:
    """Find three retinal examples spanning the supplied label set."""
    examples = []
    for label, pattern in activity_16_RETINA_EXAMPLES:
        matches = sorted((DATA_DIRECTORY / "retinamnist").glob(pattern))
        if not matches:
            raise FileNotFoundError(f"No retinal example matches {pattern!r}.")
        examples.append({"expected_label": label, "image_path": matches[0]})
    random.Random(activity_16_RANDOM_SEED).shuffle(examples)
    return examples

def activity_16_save_unlabeled_grid(examples: list[dict[str, str | Path]]) -> Path:
    """Save the images in shuffled order using neutral captions."""
    figure, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 4))
    for index, (axis, example) in enumerate(zip(axes, examples), start=1):
        with PILImage.open(Path(example["image_path"])) as image:
            axis.imshow(image)
        axis.set_title(f"Example {index}")
        axis.axis("off")
    figure.tight_layout()
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_16_retina_prioritization_examples.png"
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_16_prioritize_for_review(examples: list[dict[str, str | Path]]) -> str:
    """Ask the model to order images for review and explain its uncertainty."""
    content: list[dict[str, str | dict[str, str]]] = [
        {
            "type": "text",
            "text": (
                "Order the three images for clinician review, most to least, using "
                "only visible changes, image quality, and uncertainty. Output one "
                "line in this format only: REVIEW ORDER: Example N > Example N > "
                "Example N. Use each example once. Do not name diagnoses, causes, "
                "systemic diseases, or treatment. This is unvalidated teaching "
                "output, not clinical triage. The images are different people."
            ),
        }
    ]
    for index, example in enumerate(examples, start=1):
        image_path = Path(example["image_path"])
        mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
        encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
        content.append({"type": "text", "text": f"Example {index}"})
        content.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:{mime_type};base64,{encoded_image}"},
            }
        )
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        max_tokens=200,
        messages=[{"role": "user", "content": content}],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_16_parse_priority_lines(response_text: str) -> list[dict[str, str | int]]:
    """Parse a compact review order, with support for labeled priority lines."""
    order_match = re.search(r"REVIEW ORDER\s*:\s*(.+)", response_text, re.I)
    if order_match:
        example_numbers = [
            int(number)
            for number in re.findall(r"Example\s*(\d+)", order_match.group(1), re.I)
        ]
        return [
            {"priority": priority, "example": example, "reason": ""}
            for priority, example in enumerate(example_numbers, start=1)
        ]

    parsed = []
    pattern = re.compile(
        r"PRIORITY\s*(\d+)\s*:\s*Example\s*(\d+)\s*\|\s*REASON\s*:\s*(.*)",
        re.I,
    )
    for line in response_text.splitlines():
        match = pattern.search(line.strip())
        if match:
            parsed.append(
                {
                    "priority": int(match.group(1)),
                    "example": int(match.group(2)),
                    "reason": match.group(3).strip(),
                }
            )
    return parsed

def activity_16_validate_priority_lines(parsed: list[dict[str, str | int]]) -> list[str]:
    """Flag incomplete or duplicate ranks; this check does not assess reasons."""
    issues = []
    priorities = [int(item["priority"]) for item in parsed]
    examples = [int(item["example"]) for item in parsed]
    if priorities != [1, 2, 3]:
        issues.append("Could not parse a complete ordered list of three priorities.")
    if sorted(examples) != [1, 2, 3]:
        issues.append("The response did not assign each of Examples 1–3 exactly once.")
    return issues

def activity_16_save_prioritization_response(
    response_text: str,
    parsed_priorities: list[dict[str, str | int]],
    validation_issues: list[str],
) -> Path:
    """Save raw text and parsed rank lines for human review."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_16_prioritization_response.json"
    result = {
        "raw_response": response_text,
        "parsed_priorities": parsed_priorities,
        "format_issues": validation_issues,
        "caveat": "Unvalidated teaching output; not a clinical triage recommendation.",
    }
    output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(response_text)
    if validation_issues:
        print("Priority format needs review:")
        for issue in validation_issues:
            print(f"- {issue}")
    print(f"Saved response to: {output_path}")
    return output_path


In [ ]:
examples_16 = activity_16_load_examples()
preview_16 = activity_16_save_unlabeled_grid(examples_16)
display(Image(filename=str(preview_16)))


In [ ]:
raw_16 = activity_16_prioritize_for_review(examples_16)
print(raw_16)


In [ ]:
parsed_16 = activity_16_parse_priority_lines(raw_16)
issues_16 = activity_16_validate_priority_lines(parsed_16)
activity_16_save_prioritization_response(raw_16, parsed_16, issues_16)


**Pause:** Discuss the model's order and any format issues. When ready, reveal the dataset labels in the next cell. These labels are not a reference for clinical urgency.

In [ ]:
labels_16 = {"155": "No diabetic retinopathy", "181": "Moderate diabetic retinopathy", "165": "Proliferative diabetic retinopathy"}
print("Dataset labels (reveal after reviewing the model's ordering):")
for index, example in enumerate(examples_16, start=1):
    image_id = Path(example["image_path"]).name.split("_", maxsplit=1)[0]
    print(f"Example {index}: {labels_16[image_id]}")


## Activity 17 — List possible interpretations and evidence needed

Treating interpretations as possibilities rather than diagnoses can help avoid premature conclusions. Identifying what additional evidence would be needed also makes clear what cannot be determined from the image alone.


In [ ]:
activity_17_IMAGE_PATTERN = "pneumoniamnist/11_*.jpg"

activity_17_PROMPT = (
    "Describe visible observations, then list a small number of possible "
    "interpretations as hypotheses rather than diagnoses. For each, say what "
    "additional evidence or clinical information would help assess it. State "
    "uncertainty and do not recommend treatment. This is an educational exercise."
)

def activity_17_find_example_image() -> Path:
    """Find one prepared chest X-ray example."""
    matches = sorted(DATA_DIRECTORY.glob(activity_17_IMAGE_PATTERN))
    if not matches:
        raise FileNotFoundError(f"No image matches {activity_17_IMAGE_PATTERN!r}.")
    return matches[0]

def activity_17_save_image_preview(image_path: Path) -> Path:
    """Copy the image under a label-neutral teaching filename."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_17_possible_interpretations_image.jpg"
    output_path.write_bytes(image_path.read_bytes())
    print(f"Saved image preview to: {output_path}")
    return output_path

def activity_17_generate_possible_interpretations(image_path: Path, prompt: str = activity_17_PROMPT) -> str:
    """Ask MedGemma for possibilities and the evidence needed to assess them."""
    mime_type = mimetypes.guess_type(image_path.name)[0] or "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{encoded_image}"
                        },
                    },
                ],
            }
        ],
    )
    answer = response.choices[0].message.content or ""
    if "<unused95>" in answer:
        answer = answer.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return answer or "The model returned an empty response."

def activity_17_save_interpretation_response(response_text: str) -> Path:
    """Save the response for review and discussion."""
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_17_possible_interpretations.txt"
    output_path.write_text(response_text + "\n", encoding="utf-8")
    print(response_text)
    print(f"Saved response to: {output_path}")
    return output_path


In [ ]:
image_17 = activity_17_find_example_image()
preview_17 = activity_17_save_image_preview(image_17)
display(Image(filename=str(preview_17)))


In [ ]:
response_17 = activity_17_generate_possible_interpretations(image_17)
activity_17_save_interpretation_response(response_17)


# Workshop 2: Clinical note and structured-data activities

The remaining activities use plain-text clinical notes. They illustrate a pipeline from extraction through normalization and audit. Model output is kept alongside locally parsed fields so that you can inspect both rather than treating the parsed result as automatically correct.

In these activities, the model extracts information from a note. The notebook may then sort the response into fields or check quoted text against the note. Compare each result with the model’s original response and the note: these steps do not confirm clinical accuracy.


## Activity 18 — Extract clinical facts with source quotes

Quotes make it easier to check whether an extracted fact is supported by the note. “Not stated” means the note does not document the information; it does not mean the information is absent.

The model returns labeled lines with quotes from the note. A script sorts those lines into fields and keeps the original response alongside them, so you can compare what the model said with what the script collected.


In [ ]:
activity_18_CLINICAL_NOTE = """
Visit note: On April 10, 2025, the patient reports a dry cough for
three days and a temperature of 38.1 C yesterday evening. They report taking
acetaminophen as needed. Allergy information was not discussed in this note.
""".strip()

def activity_18_extract_clinical_facts(note: str = activity_18_CLINICAL_NOTE) -> str:
    """Ask the model for labeled plain-text facts with supporting source quotes."""
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract only facts explicitly stated in the note. Do not "
                    "diagnose or infer missing facts. For each extracted fact, "
                    "include a short exact quote copied from the note. Do not "
                    "treat undocumented information as a negative finding. Use "
                    "one plain-text line per fact with these labels: SYMPTOM:, "
                    "MEDICATION:, DATE:, OBSERVATION:, or NOT STATED:. For facts, "
                    "append | SOURCE QUOTE: followed by an exact quote. Use each "
                    "fact once, including symptom duration. For undocumented "
                    "categories, write only NOT STATED: category. Follow this "
                    "format: DATE: 2025-04-10 | SOURCE QUOTE: On April 10, 2025; "
                    "SYMPTOM: dry cough for three days | SOURCE QUOTE: reports a "
                    "dry cough for three days; OBSERVATION: temperature 38.1 C | "
                    "SOURCE QUOTE: temperature of 38.1 C yesterday evening; "
                    "MEDICATION: acetaminophen as needed | SOURCE QUOTE: taking "
                    "acetaminophen as needed; NOT STATED: allergies. Use only "
                    "these labels and do not return JSON."
                ),
            },
            {"role": "user", "content": note},
        ],
    )
    response_text = response.choices[0].message.content or ""
    if "<unused95>" in response_text:
        response_text = response_text.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return response_text or "The model returned an empty response."

def activity_18_parse_labeled_facts(response_text: str) -> dict[str, object]:
    """Parse category labels and source quotes while preserving the raw text."""
    category_map = {
        "symptom": "symptoms",
        "medication": "medications",
        "date": "dates",
        "observation": "observations",
    }
    parsed: dict[str, object] = {
        "symptoms": [],
        "medications": [],
        "dates": [],
        "observations": [],
        "not_stated": [],
        "unparsed_lines": [],
        "raw_response": response_text,
    }
    for line in response_text.splitlines():
        line = line.strip().lstrip("-*• ").strip()
        match = re.match(r"^(SYMPTOM|MEDICATION|DATE|OBSERVATION|NOT STATED)\s*:\s*(.*)$", line, re.I)
        if not match:
            if line:
                parsed["unparsed_lines"].append(line)
            continue

        category = match.group(1).casefold()
        value_and_quote = match.group(2).strip()
        if category == "not stated":
            category_text = re.split(
                r"\|\s*SOURCE QUOTE\s*:", value_and_quote, maxsplit=1, flags=re.I
            )[0]
            parsed["not_stated"].append(category_text.strip().rstrip(".; "))
            continue

        pieces = re.split(r"\|\s*SOURCE QUOTE\s*:\s*", value_and_quote, maxsplit=1, flags=re.I)
        fact = {
            "value": pieces[0].strip(),
            "source_quote": pieces[1].strip().strip('"').rstrip(".; ")
            if len(pieces) > 1
            else "",
        }
        parsed[category_map[category]].append(fact)
    return parsed

def activity_18_save_extraction(
    note: str, response_text: str, parsed_extraction: dict[str, object]
) -> Path:
    """Save the raw labeled response beside fields parsed from its labels."""
    result = {
        "source_note": note,
        "raw_model_response": response_text,
        "parsed_extraction": parsed_extraction,
    }
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_18_clinical_fact_extraction.json"
    output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))
    print(f"Saved extraction to: {output_path}")
    return output_path


In [ ]:
note_18 = activity_18_CLINICAL_NOTE
print(note_18)


In [ ]:
raw_18 = activity_18_extract_clinical_facts(note_18)
print(raw_18)


In [ ]:
parsed_18 = activity_18_parse_labeled_facts(raw_18)
activity_18_save_extraction(note_18, raw_18, parsed_18)


## Activity 19 — Preserve denials and timing

Whether a symptom is present or denied, and whether a medication was used in the past or is current, changes the meaning of a record. Missing information is also different from a negative finding.

The model labels facts as present or denied and as current or historical. A script organizes those labels into fields; compare its result with the original response to check that these distinctions remain intact.


In [ ]:
activity_19_CLINICAL_NOTE = """
Follow-up note: The patient reports a cough for four days and denies
shortness of breath today. They took amoxicillin in 2022 but are not taking it
now. No allergy information is documented in this note.
""".strip()

def activity_19_extract_negation_and_time(note: str = activity_19_CLINICAL_NOTE) -> str:
    """Ask for labeled plain-text facts that preserve denial and temporal status."""
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract only note-supported facts. Use status present for an "
                    "affirmed fact, denied for an explicit denial, and not_stated "
                    "only for information the note says is undocumented. When a "
                    "medication is described as taken in the past but not taken "
                    "now, represent both facts separately. Use one plain-text line "
                    "per fact in this format: FACT: concept | STATUS: present or "
                    "denied | TIME: current, historical, or unspecified | VALUE: "
                    "short description | SOURCE QUOTE: exact words from the note. "
                    "Use NOT STATED: category for undocumented information. Do not "
                    "return JSON. Include each of these distinctions: "
                    "FACT: cough | STATUS: present | TIME: current | VALUE: cough "
                    "for four days | SOURCE QUOTE: reports a cough for four days; "
                    "FACT: shortness of breath | STATUS: denied | TIME: current | "
                    "VALUE: shortness of breath | SOURCE QUOTE: denies shortness "
                    "of breath today; FACT: amoxicillin | STATUS: present | TIME: "
                    "historical | VALUE: took amoxicillin | SOURCE QUOTE: They took "
                    "amoxicillin in 2022; FACT: amoxicillin | STATUS: denied | "
                    "TIME: current | VALUE: not taking amoxicillin | SOURCE QUOTE: "
                    "not taking it now; NOT STATED: allergies. Put each example "
                    "on its own line."
                ),
            },
            {"role": "user", "content": note},
        ],
    )
    response_text = response.choices[0].message.content or ""
    if "<unused95>" in response_text:
        response_text = response_text.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return response_text or "The model returned an empty response."

def activity_19_parse_labeled_facts(response_text: str) -> dict[str, object]:
    """Parse labeled fact rows while preserving any lines the parser misses."""
    facts = []
    not_stated = []
    unparsed_lines = []
    for line in response_text.splitlines():
        line = line.strip().lstrip("-*• ").strip()
        if not line:
            continue
        if re.match(r"^NOT STATED\s*:", line, re.I):
            category = re.split(
                r"^NOT STATED\s*:\s*", line, maxsplit=1, flags=re.I
            )[-1].rstrip(".; ")
            if category not in not_stated:
                not_stated.append(category)
            continue
        if not re.match(r"^FACT\s*:", line, re.I):
            unparsed_lines.append(line)
            continue

        fact = {}
        for segment in line.split("|"):
            if ":" not in segment:
                continue
            key, value = segment.split(":", maxsplit=1)
            key = key.casefold().strip()
            value = value.strip().strip('"')
            if key in {"fact", "concept"}:
                fact["concept"] = value
            elif key == "status":
                fact["status_as_returned"] = value
                fact["status"] = "present" if value.casefold() == "historical" else value
            elif key in {"time", "temporality"}:
                fact["temporality"] = (
                    "historical" if value.casefold() == "past" else value
                )
            elif key == "value":
                fact["value"] = value
            elif key in {"source quote", "quote", "evidence"}:
                if value.casefold() not in {"none", "not stated", "n/a"}:
                    fact["source_quote"] = value.rstrip(".; ")
        if fact:
            facts.append(fact)
            if (
                fact.get("status", "").casefold() == "not_stated"
                or fact.get("status_as_returned", "").casefold() == "not_stated"
            ):
                category = fact.get("concept", "unspecified")
                if category not in not_stated:
                    not_stated.append(category)
        else:
            unparsed_lines.append(line)
    return {
        "facts": facts,
        "not_stated": not_stated,
        "unparsed_lines": unparsed_lines,
        "raw_response": response_text,
    }

def activity_19_save_extraction(
    note: str, response_text: str, parsed_extraction: dict[str, object]
) -> Path:
    """Save the raw labeled text alongside locally parsed fields."""
    result = {
        "source_note": note,
        "raw_model_response": response_text,
        "parsed_extraction": parsed_extraction,
    }
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_19_negation_temporality_extraction.json"
    output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"Saved extraction to: {output_path}")
    return output_path


In [ ]:
note_19 = activity_19_CLINICAL_NOTE
print(note_19)


In [ ]:
raw_19 = activity_19_extract_negation_and_time(note_19)
print(raw_19)


In [ ]:
parsed_19 = activity_19_parse_labeled_facts(raw_19)
print(json.dumps(parsed_19, indent=2))
activity_19_save_extraction(note_19, raw_19, parsed_19)


## Activity 20 — Organize facts into standard fields

Consistent fields can make information easier to review and compare. Preserving the values and units as written helps prevent changes in meaning; organizing the response does not verify that the model interpreted the note correctly.

The model returns labeled lines for dates, measurements, and medications. A script puts them into fields while retaining the values and units as written; it does not convert units or verify the model’s interpretation.


In [ ]:
activity_20_CLINICAL_NOTE = """
Visit note dated April 10, 2025: temperature 100.6 degrees F, pulse
92 beats per minute, and blood pressure 126/78 mmHg. The patient says they take
acetaminophen 650 mg by mouth as needed.
""".strip()

def activity_20_normalize_facts(note: str = activity_20_CLINICAL_NOTE) -> str:
    """Ask for labeled plain text that maps note content to normalized fields."""
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Map only explicitly stated note content into the requested "
                    "fields. Format the date as YYYY-MM-DD. Use the "
                    "provided concept values temperature, pulse, or blood_pressure. "
                    "For each observation, preserve the written number in "
                    "value_as_written, put a normalized numeric string in "
                    "normalized_value, and retain its stated unit; do not convert "
                    "units. For medication, report only stated name, dose, route, "
                    "and frequency; use 'not stated' where needed. Include an "
                    "exact source quote for every observation and medication. "
                    "Return labeled plain-text lines, not JSON. Use one record per "
                    "line in these formats:\n"
                    "ENCOUNTER DATE: 2025-04-10\n"
                    "OBSERVATION: temperature | ORIGINAL VALUE: 100.6 | "
                    "NORMALIZED VALUE: 100.6 | UNIT: degrees F | SOURCE QUOTE: "
                    "temperature 100.6 degrees F\n"
                    "MEDICATION: acetaminophen | DOSE: 650 mg | ROUTE: by mouth | "
                    "FREQUENCY: as needed | SOURCE QUOTE: acetaminophen 650 mg by "
                    "mouth as needed\n"
                    "NOT STATED: allergies"
                ),
            },
            {"role": "user", "content": note},
        ],
    )
    response_text = response.choices[0].message.content or ""
    if "<unused95>" in response_text:
        response_text = response_text.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return response_text or "The model returned an empty response."

def activity_20_parse_normalized_fields(response_text: str) -> dict[str, object]:
    """Parse the labeled lines without requiring the model to emit JSON."""
    normalized: dict[str, object] = {
        "encounter_date": "",
        "observations": [],
        "medications": [],
        "not_stated": [],
        "unparsed_lines": [],
        "raw_response": response_text,
    }
    for line in response_text.splitlines():
        line = line.strip().lstrip("-*• ").strip()
        if not line:
            continue
        if re.match(r"^ENCOUNTER DATE\s*:", line, re.I):
            normalized["encounter_date"] = re.split(
                r"^ENCOUNTER DATE\s*:\s*", line, maxsplit=1, flags=re.I
            )[-1].strip()
            continue
        if re.match(r"^NOT STATED\s*:", line, re.I):
            normalized["not_stated"].append(
                re.split(r"^NOT STATED\s*:\s*", line, maxsplit=1, flags=re.I)[-1]
            )
            continue

        segments = [segment.strip() for segment in line.split("|")]
        if not segments or ":" not in segments[0]:
            normalized["unparsed_lines"].append(line)
            continue
        kind, first_value = segments[0].split(":", maxsplit=1)
        kind = kind.casefold().strip()
        record_key = "concept" if kind == "observation" else "name"
        record: dict[str, str] = {record_key: first_value.strip()}
        field_map = {
            "original value": "value_as_written",
            "normalized value": "normalized_value",
            "unit": "unit",
            "dose": "dose_as_written",
            "route": "route",
            "frequency": "frequency",
            "source quote": "source_quote",
        }
        for segment in segments[1:]:
            if ":" not in segment:
                continue
            key, value = segment.split(":", maxsplit=1)
            normalized_key = key.casefold().strip()
            if normalized_key in field_map:
                record[field_map[normalized_key]] = value.strip().strip('"')
        if kind == "observation":
            normalized["observations"].append(record)
        elif kind == "medication":
            normalized["medications"].append(record)
        else:
            normalized["unparsed_lines"].append(line)
    return normalized

def activity_20_display_normalized_note(
    note: str, response_text: str, normalized: dict[str, object]
) -> None:
    """Display source text, raw labeled output, and locally parsed fields."""
    print(f"Source note:\n{note}\n")
    print(f"Raw labeled response:\n{response_text}\n")
    print(json.dumps(normalized, indent=2))

def activity_20_save_normalized_note(
    note: str, response_text: str, normalized: dict[str, object]
) -> Path:
    """Save raw text and locally parsed fields for comparison."""
    result = {
        "source_note": note,
        "raw_model_response": response_text,
        "parsed_normalized_fields": normalized,
    }
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_20_normalized_clinical_note.json"
    output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"Saved normalized result to: {output_path}")
    return output_path


In [ ]:
note_20 = activity_20_CLINICAL_NOTE
print(note_20)


In [ ]:
raw_20 = activity_20_normalize_facts(note_20)
print(raw_20)


In [ ]:
parsed_20 = activity_20_parse_normalized_fields(raw_20)
activity_20_display_normalized_note(note_20, raw_20, parsed_20)
activity_20_save_normalized_note(note_20, raw_20, parsed_20)


## Activity 21 — Check extracted data against the source note

Automated text checks can flag missing or unsupported quotes, but they cannot determine whether the extracted information is clinically correct or complete. Review any flags alongside the original note.

The script checks whether quotes appear in the note and whether several expected phrases were included. These text checks can flag possible omissions or unsupported quotes, but they cannot tell whether a fact was interpreted correctly.


In [ ]:
activity_21_CLINICAL_NOTE = """
Visit note: The patient reports a dry cough for three days and fever
of 38.1 C yesterday evening. They take acetaminophen as needed. The note states
that allergy information was not discussed.
""".strip()

activity_21_REQUIRED_SOURCE_PHRASES = [
    "dry cough",
    "fever of 38.1 C",
    "acetaminophen as needed",
]

def activity_21_extract_for_audit(note: str = activity_21_CLINICAL_NOTE) -> str:
    """Ask for labeled plain-text facts with quotes that can be checked locally."""
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=180.0)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract explicit facts as plain-text lines, not JSON. For each "
                    "fact use this format: FACT: category | VALUE: short value | "
                    "SOURCE QUOTE: exact words copied from the note. For information "
                    "explicitly described as not discussed, use NOT STATED: category. "
                    "Include the dry cough, fever, acetaminophen, and undocumented "
                    "allergy information. Do not infer diagnoses."
                ),
            },
            {"role": "user", "content": note},
        ],
    )
    response_text = response.choices[0].message.content or ""
    if "<unused95>" in response_text:
        response_text = response_text.rsplit("<unused95>", maxsplit=1)[-1].strip()
    return response_text or "The model returned an empty response."

def activity_21_parse_labeled_extraction(response_text: str) -> dict[str, Any]:
    """Parse FACT and NOT STATED lines while retaining any unparsed text."""
    facts = []
    not_stated = []
    unparsed_lines = []
    for line in response_text.splitlines():
        line = line.strip().lstrip("-*• ").strip()
        if not line:
            continue
        if re.match(r"^NOT STATED\s*:", line, re.I):
            category_text = re.split(
                r"^NOT STATED\s*:\s*", line, maxsplit=1, flags=re.I
            )[-1]
            not_stated.append(category_text.split("|", maxsplit=1)[0].strip().casefold())
            continue
        if not re.match(r"^FACT\s*:", line, re.I):
            unparsed_lines.append(line)
            continue

        fact = {}
        for segment in line.split("|"):
            if ":" not in segment:
                continue
            key, value = segment.split(":", maxsplit=1)
            normalized_key = key.casefold().strip()
            value = value.strip().strip('"').rstrip(".; ")
            if normalized_key in {"fact", "category", "field"}:
                fact["field"] = value
            elif normalized_key == "value":
                fact["value"] = value
            elif normalized_key in {"source quote", "quote", "evidence"}:
                fact["source_quote"] = value
        if fact:
            facts.append(fact)
        else:
            unparsed_lines.append(line)
    return {
        "facts": facts,
        "not_stated": not_stated,
        "unparsed_lines": unparsed_lines,
        "raw_response": response_text,
    }

def activity_21_audit_extraction(
    note: str, extraction: dict[str, Any] | str
) -> list[str]:
    """Flag quotes not found in the note and expected phrases not extracted."""
    if not isinstance(extraction, dict):
        return ["Response was not parsed into labeled fact fields."]

    facts = extraction.get("facts")
    if not isinstance(facts, list):
        return ["No parsed FACT lines were available to audit."]

    normalized_note = " ".join(note.casefold().split())
    issues = []
    extracted_quotes = []
    for index, fact in enumerate(facts, start=1):
        if not isinstance(fact, dict):
            issues.append(f"Fact {index} is not an object.")
            continue
        source_quote = fact.get("source_quote")
        if not isinstance(source_quote, str) or not source_quote.strip():
            issues.append(f"Fact {index} has no source quote.")
            continue
        extracted_quotes.append(" ".join(source_quote.casefold().split()))
        if " ".join(source_quote.casefold().split()) not in normalized_note:
            issues.append(f"Fact {index} quote is not an exact note substring.")

    for required_phrase in activity_21_REQUIRED_SOURCE_PHRASES:
        normalized_phrase = " ".join(required_phrase.casefold().split())
        if not any(normalized_phrase in quote for quote in extracted_quotes):
            issues.append(f"Expected source information may be missing: {required_phrase}.")
    return issues

def activity_21_save_audit_result(
    note: str,
    response_text: str,
    extraction: dict[str, Any],
    issues: list[str],
) -> Path:
    """Save model output and simple audit flags for human follow-up."""
    result = {
        "source_note": note,
        "raw_model_response": response_text,
        "extraction": extraction,
        "audit_status": "needs human review" if issues else "basic checks passed",
        "audit_issues": issues,
        "limitation": (
            "Substring checks do not establish clinical correctness or completeness; "
            "review the source and extraction manually."
        ),
    }
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIRECTORY / "activity_21_extraction_audit.json"
    output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))
    print(f"Saved audit to: {output_path}")
    return output_path


In [ ]:
note_21 = activity_21_CLINICAL_NOTE
print(note_21)


In [ ]:
raw_21 = activity_21_extract_for_audit(note_21)
print(raw_21)


In [ ]:
parsed_21 = activity_21_parse_labeled_extraction(raw_21)
issues_21 = activity_21_audit_extraction(note_21, parsed_21)
audit_path_21 = activity_21_save_audit_result(note_21, raw_21, parsed_21, issues_21)
print("Audit flags:", issues_21 or "Basic checks passed; human review is still needed.")
print("Audit file:", audit_path_21)


## Wrap-up and discussion

- Which model statements were directly supported by their source material?
- Where did the model show uncertainty, and where did it sound more certain than the evidence justified?
- What can go wrong when an image and a record are not known to belong to the same person?
- Which parsing or audit checks were useful, and what could they not establish?
- What would you want a human reviewer to verify before using any generated text?

All outputs in this notebook are teaching examples. They require source review and must not be used to make clinical decisions.
